In [1]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
from pypdf import PdfReader
from dotenv import load_dotenv

# Load OPENAI_API_KEY from a local .env file (see .env.example) - never hard-code a key
load_dotenv()

client = OpenAI()

class PaperData(BaseModel):
    title: str = Field(description="The title of the paper")
    abstract_summary: List[str] = Field(description="One sentence summary of the abstract")
    intro_backgrounds_summary: str = Field(description="One sentence summary of the intro and backgrounds sections")
    methods_summary: str = Field(description="One sentence summary of the methods section")
    results_summary: str = Field(description="One sentence summary of the results section")
    discussion_summary: str = Field(description="One sentence summary of the discussion section")
    

def load_pdf_text(file_path):
    '''Loads text from a PDF file.'''
    # creating a pdf reader object
    reader = PdfReader(file_path)

    # extracting text from page
    text = "\n\n".join([page.extract_text() for page in reader.pages])
    
    return text    

In [2]:
import os, urllib.request

def download_pdf(url, dest):
    """Download a PDF to `dest` if it is not already there (cross-platform; no wget)."""
    if os.path.exists(dest):
        print(f"Already present: {dest}")
        return
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(dest, "wb") as f:
        f.write(r.read())
    print(f"Downloaded: {dest}")

download_pdf("https://arxiv.org/pdf/2510.26493", "../assets/paper.pdf")

Already present: ../assets/paper.pdf


In [3]:
paper_raw_text = load_pdf_text("../assets/paper.pdf")

response = client.responses.parse(
    model="gpt-5.6-luna",
    input=f"extract the information from the paper: {paper_raw_text}",
    text_format=PaperData
)

print(response)

ParsedResponse[PaperData](id='resp_087235512bd38bda006aafe98a592087d2bfd0c4e0d20ca4b5', created_at=1789913482.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.6-luna', object='response', output=[ResponseReasoningItem(id='rs_087235512bd38bda006aafe98b115487d2a0a964b74ba89714', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqr-mOjDuRU-YD1MlwO8ublwEtDriVwf2O8ZEsF36JYaIslE_E_nvd55V2PHv-kCY59JqLoKVDRSPL5aFtjNkXMikMN6GU5-lKHI1XEoGLs_TTR6txnBEe1nxxMXUK3MrGmDgPbc7NIza6t8LWj0wO_Ry3ugaqWtsaYvJB-pbwnw7r4NsgqXfr3SJXv0z4G_OWv5hXZRLAlPcYl5E2hBJF80pHAja7bPDDmXJN-VuVnYHW1iMfmRnzHR6GV7dHY-g9zhBeVvj2chPvPHdxyyvhZfbdJh8Y9WPqccc1tP-m2yZGfRXd0bvNLoTsZ0Y43gbMQEZzagiiU0TBTlH4SZTCFUKYITT_3DfHr70m2tAG2aH4a5s75xISJVO4TszO7AtcHP_sRTi0OnCQlSLgENpC3fw9ctYaSj_COiIws2JwKlw6Vyh5XZfW9KiloyC2IfpTcs8fVbTkbrXmAF6XxJmP45cKWXI36V3YAkcQJa8cjBduh1p3BC1XMc8UmnNyouXkBt7BbeFrZ8IV_fvy8xokFhYINscWg6_9S9wi0UN99QDbOtnazpaPPoiNPtOYTBw3bsAP6il724q9Upf_eEhnUvkfm7F4smpRnSPo1JHCf4

In [4]:
response.output_parsed

PaperData(title='Context Engineering 2.0: The Context of Context Engineering', abstract_summary=['The paper defines context engineering as the systematic collection, storage, management, and use of contextual information to improve machine understanding, traces its roots from early human–computer interaction to modern intelligent agents, and proposes a four-stage evolution toward human-level and superhuman context processing.'], intro_backgrounds_summary='The introduction argues that context engineering is a long-standing effort to reduce the gap between human intent and machine understanding, extending beyond recent prompt and agent techniques to include decades of work in HCI, ubiquitous computing, and context-aware systems.', methods_summary='The paper uses a conceptual and historical analysis, formalizing entities, interactions, context, and context engineering mathematically, comparing engineering practices across eras, and organizing design considerations around context collectio

In [5]:
from IPython.display import Markdown, display

md = f"""
# {response.output_parsed.title}

## Abstract Summary
{'; '.join(response.output_parsed.abstract_summary)}

## Introduction & Backgrounds Summary
{response.output_parsed.intro_backgrounds_summary}

## Methods Summary
{response.output_parsed.methods_summary}

## Results Summary
{response.output_parsed.results_summary}

## Discussion Summary
{response.output_parsed.discussion_summary}
"""

display(Markdown(md))


# Context Engineering 2.0: The Context of Context Engineering

## Abstract Summary
The paper defines context engineering as the systematic collection, storage, management, and use of contextual information to improve machine understanding, traces its roots from early human–computer interaction to modern intelligent agents, and proposes a four-stage evolution toward human-level and superhuman context processing.

## Introduction & Backgrounds Summary
The introduction argues that context engineering is a long-standing effort to reduce the gap between human intent and machine understanding, extending beyond recent prompt and agent techniques to include decades of work in HCI, ubiquitous computing, and context-aware systems.

## Methods Summary
The paper uses a conceptual and historical analysis, formalizing entities, interactions, context, and context engineering mathematically, comparing engineering practices across eras, and organizing design considerations around context collection, management, and usage.

## Results Summary
The analysis presents a four-era framework—primitive computation, agent-centric intelligence, human-level intelligence, and speculative superhuman intelligence—and identifies strategies for multimodal collection, layered memory, context abstraction, agent collaboration, context selection, proactive need inference, and lifelong context preservation.

## Discussion Summary
The discussion argues that future systems require a semantic operating system for dynamically storing, abstracting, selecting, sharing, correcting, and explaining lifelong context while addressing scalability, processing degradation, instability, evaluation, privacy, and the limitations of current models.


In [6]:
import pandas as pd
import os

def upsert_paper_to_csv(paper_data, csv_path="papers_database.csv", key_column="title"):
    """
    Inserts or updates a paper's data in a CSV file database. If the file does not exist,
    it creates it. If the file exists, it updates the entry if the key exists, or appends 
    the new paper if not.

    Parameters:
        paper_data: an object with attributes corresponding to extracted fields
        csv_path: output file path for CSV
        key_column: column name to use as unique identifier (default: "title")
    """
    # Prepare data dictionary, flattening abstract_summary if it's a list
    paper_dict = {
        "title": getattr(paper_data, "title", ""),
        "abstract_summary": "; ".join(paper_data.abstract_summary) if isinstance(getattr(paper_data, "abstract_summary", ""), list) else getattr(paper_data, "abstract_summary", ""),
        "intro_backgrounds_summary": getattr(paper_data, "intro_backgrounds_summary", ""),
        "methods_summary": getattr(paper_data, "methods_summary", ""),
        "results_summary": getattr(paper_data, "results_summary", ""),
        "discussion_summary": getattr(paper_data, "discussion_summary", ""),
    }

    # If CSV exists, load and update if key matches; otherwise create new
    if os.path.isfile(csv_path):
        df = pd.read_csv(csv_path)
        # Check if key (e.g., title) already in db
        if paper_dict[key_column] in df[key_column].values:
            # Update existing entry
            df.loc[df[key_column] == paper_dict[key_column], list(paper_dict.keys())] = list(paper_dict.values())
            updated = True
        else:
            # Append if not found
            df = pd.concat([df, pd.DataFrame([paper_dict])], ignore_index=True)
            updated = False
    else:
        # Create new dataframe
        df = pd.DataFrame([paper_dict])
        updated = False

    df.to_csv(csv_path, index=False)
    if updated:
        print(f"Updated entry in '{csv_path}' for {key_column}='{paper_dict[key_column]}'")
    else:
        print(f"Added new entry to '{csv_path}' for {key_column}='{paper_dict[key_column]}'")

# Example usage:
upsert_paper_to_csv(response.output_parsed)

Added new entry to 'papers_database.csv' for title='Context Engineering 2.0: The Context of Context Engineering'


In [7]:
import pandas as pd

df = pd.read_csv("../assets/papers_database.csv")

df

,title,abstract_summary,intro_backgrounds_summary,methods_summary,results_summary,discussion_summary
0,Context Engineering 2.0: The Context of Contex...,The paper defines and situates context enginee...,The introduction motivates context engineering...,The paper formalizes context with mathematical...,"Rather than empirical experiments, the paper p...",The discussion highlights key challenges for l...
1,Efﬁcient Estimation of Word Representations in...,The paper introduces two efficient model archi...,The authors motivate moving beyond atomic word...,They propose two log-linear architectures—CBOW...,"Experimentally, Skip-gram yields the best sema...","The paper concludes that very simple, computat..."


In [8]:
download_pdf("https://arxiv.org/pdf/1301.3781", "../assets/paper3.pdf")

Already present: ../assets/paper3.pdf


In [9]:
def extract_paper_data_workflow(paper_path: str):
    paper_raw_text = load_pdf_text(paper_path)

    response = client.responses.parse(
        model="gpt-5.6-luna",
        input=f"extract the information from the paper: {paper_raw_text}",
        text_format=PaperData
    )
    
    output_parsed = response.output_parsed

    print(output_parsed)
    
    upsert_paper_to_csv(output_parsed)
    
    return output_parsed

extract_paper_data_workflow("../assets/paper3.pdf")

title='Efficient Estimation of Word Representations in Vector Space' abstract_summary=['The paper introduces the CBOW and Skip-gram architectures for efficiently learning high-quality continuous word vectors from very large corpora, demonstrating improved semantic and syntactic accuracy at substantially lower computational cost than earlier neural language models.'] intro_backgrounds_summary='The paper motivates distributed word representations as an alternative to atomic word indices and reviews prior neural, recurrent, and log-bilinear approaches, emphasizing the need for scalable methods that learn linear semantic and syntactic relationships from billions of words.' methods_summary='The authors compare feedforward NNLMs, RNNLMs, CBOW, and Skip-gram models trained with stochastic or distributed asynchronous gradient descent and hierarchical softmax, evaluating word vectors through analogy-style semantic and syntactic relationship tests and a sentence-completion benchmark.' results_su

PaperData(title='Efficient Estimation of Word Representations in Vector Space', abstract_summary=['The paper introduces the CBOW and Skip-gram architectures for efficiently learning high-quality continuous word vectors from very large corpora, demonstrating improved semantic and syntactic accuracy at substantially lower computational cost than earlier neural language models.'], intro_backgrounds_summary='The paper motivates distributed word representations as an alternative to atomic word indices and reviews prior neural, recurrent, and log-bilinear approaches, emphasizing the need for scalable methods that learn linear semantic and syntactic relationships from billions of words.', methods_summary='The authors compare feedforward NNLMs, RNNLMs, CBOW, and Skip-gram models trained with stochastic or distributed asynchronous gradient descent and hierarchical softmax, evaluating word vectors through analogy-style semantic and syntactic relationship tests and a sentence-completion benchmark

In [10]:
import pandas as pd


df = pd.read_csv("../assets/papers_database.csv")
df

,title,abstract_summary,intro_backgrounds_summary,methods_summary,results_summary,discussion_summary
0,Context Engineering 2.0: The Context of Contex...,The paper defines and situates context enginee...,The introduction motivates context engineering...,The paper formalizes context with mathematical...,"Rather than empirical experiments, the paper p...",The discussion highlights key challenges for l...
1,Efﬁcient Estimation of Word Representations in...,The paper introduces two efficient model archi...,The authors motivate moving beyond atomic word...,They propose two log-linear architectures—CBOW...,"Experimentally, Skip-gram yields the best sema...","The paper concludes that very simple, computat..."


In [11]:
tool_schema = {
    "type": "function",
    "name": "extract_paper_data_workflow",
    "description": "Extracts data from a paper",
    "parameters": {
        "type": "object",
        "properties": {
            "paper_path": {
                "type": "string",
                "description": "The path to the paper"
            }
        },
        "required": ["paper_path"],
    }
}

tool_schema

{'type': 'function',
 'name': 'extract_paper_data_workflow',
 'description': 'Extracts data from a paper',
 'parameters': {'type': 'object',
  'properties': {'paper_path': {'type': 'string',
    'description': 'The path to the paper'}},
  'required': ['paper_path']}}

In [12]:
download_pdf("https://arxiv.org/pdf/2412.14161", "./test_agent_paper.pdf")

Already present: ./test_agent_paper.pdf


In [13]:
response = client.responses.create(
    model="gpt-5.6-luna",
    instructions="You are an extraction agent, users will give you paths to files for pdfs\
                  and you will extract the data from the paper using the:\
                  extract_paper_data function.",
    tools=[tool_schema],
    input="Extract data from this paper: ./test_agent_paper.pdf"
)

response.output_text

''

In [14]:
response.output

[ResponseReasoningItem(id='rs_0c4378af5277b1ea006aafe993c4d087d29bb8873a7af047b7', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqr-mUxCenvv1GdtKQJ7G5b3c9EORUddry28qmHnoqYfoSGhwkGnQQ3eu90euzU6R19peOornc0K3LVHD1mNIOljIyYy_eyoOQiVddRilUou7iuFLiqtCh92ApkxF8AHVAqgzxZCbmdvb4IeQGjupDdLO0mCxYb8iRCvmB69PuLQpdSZPWygjebafs5RFbmHYzUlDuKOKAdwkq5hSjOt0yOabXULDJ1dAT5EDKxIR3qGJ4UQLo7ZSU_-UXv1LOUA1EjYX_U42hp6ws_WUE8vh_JbWdVdh0QrskOnYC6N9Klhqv07uD2V2JkUqAekWddCn2ZJyl2emaB7MXKn-vZcNyW0dabprcuCZTBqyXazlT08RPhbKr9wnJPToGcizdk3Y7d5A_vaKYueepdVuT_XowntB5kTF5fKXc7sPKW__UMpZP6KErBc5KJWciVZ-ybAyVkQmH1DHR9acjM5tgqb8uvwcOv5BVQ4m8ZbFB0P36h2HXjoXZr22NVyvz8jCAtdzOJeXhV5gaEhNpOgNovH8wb_r-KV_G3Xgs2bUubXCEyIJGP2fcBbw8ZffntXg_IuOAdqWWVwDq8h3J3v3IXw1mdCqLicVx9z_wlH-0_NFJ5GyqrLP1jBjaZxGdCum45Q1w1cv2rlOPWy8pqbO_3j9X192Eykrw5Yr6a9CD4n02r-D-GtugMVbribs6x84NPLGv3WgsJnty14nsy1YFhJiqvdz4jfhQxxN45D3cj06CEP9IpsxSd5NLgjJjBaew2500yqZKG0a4w_4GQPlTAZw2zgJxwhbzoomVapr1LhQPkTamU-aIVhGkZ_0gZ6yo5emx9nkrudYXNGBPzbWN